In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import DataLoader, Dataset

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

In [ ]:
class Mymodel(nn.Module):
    def __init__(self, dim_list, activation_list):
        super().__init__()
        self.num_layers = len(dim_list) - 1
        self.layers = nn.Sequential()

        for l in range(self.num_layers):
            self.layers.append(nn.ModuleList())
            self.layers[-1].append(nn.Linear(dim_list[l], dim_list[l + 1]))
            self.layers[-1].append(getattr(nn, activation_list[l])())

            # If the activation function is not ReLU or GELU, then for the
            # linear module, we do Xavier's uniform initialization
            # and all zero initialization on bias (if it is not None)
            if activation_list[l] not in ['ReLU', 'GELU']:
                init.xavier_uniform_(self.layers[l][0].weight)
                if self.layers[l][0].bias is not None:
                    init.zeros_(self.layers[l][0].bias)

    def forward(self, x):
        Y = x
        for l in range(1, self.num_layers + 1):
            Z = self.layers[l - 1][0](Y)
            Y = self.layers[l - 1][1](Z)

        return Y